In [1]:
print("Let's Start")

Let's Start


In [3]:
import pickle

df = pickle.load(open('/mnt/WindowsD/Projects/commentlens/data/interim/vectorized_df.pkl' , 'rb'))
x =df.iloc[:,:-1]
y = df.iloc[:,-1]
x.shape , y.shape

((47310, 6000), (47310,))

In [1]:
import pandas as td
from sklearn.model_selection import train_test_split
df = td.read_csv('/mnt/WindowsD/Projects/commentlens/data/interim/comments.csv' , index_col=0)

df = df.iloc[: , :2]

In [4]:
from sklearn.model_selection import train_test_split

x_train ,  x_test , y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

In [3]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

GPU available: False


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

# ==========================================
# 1. Model choose karo (sirf ye ek line badal kar koi bhi BERT/RoBERTa try kar sakte ho)
# ==========================================
MODEL_NAME = "google/muril-base-cased"
# Doosre options jo isi same code mein chalenge:
# MODEL_NAME = "distilbert-base-multilingual-cased"          # Faster & lighter
# MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"  # Social media pre-trained

# ==========================================
# 2. Data Prep (same split jo LightGBM mein tha)
# ==========================================
# df = pd.read_csv("merged_comments.csv")
x = df["comments"].astype(str)
labels = df["category"].astype(str).str.strip().str.lower()
label_mapping = {"neutral": 0, "positive": 1, "negative": 2}
y = labels.map(label_mapping)

# Pehle 20% test set alag rakho (bilkul untouched)
x_train_full, x_test, y_train_full, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# Training data se 10% validation ke liye nikalo (har epoch par check karne ke liye)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.1, random_state=42, stratify=y_train_full
)

# ==========================================
# 3. Tokenizer aur PyTorch Dataset banana
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
        )
        encoding["labels"] = int(self.labels[idx])
        return encoding

train_dataset = CommentDataset(x_train, y_train, tokenizer)
val_dataset = CommentDataset(x_val, y_val, tokenizer)
test_dataset = CommentDataset(x_test, y_test, tokenizer)

# ==========================================
# 4. Class Weights (LightGBM ke class_weight="balanced" ka BERT version)
# ==========================================
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=y_train.to_numpy(),
)
class_weights_tensor = torch.tensor(weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        model_inputs = {k: v for k, v in inputs.items() if k != "labels"}
        outputs = model(**model_inputs)
        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fn(logits, labels.to(logits.device))
        return (loss, outputs) if return_outputs else loss

# ==========================================
# 5. Model Load + Metrics + Training Settings
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="./bert_commentlens",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=100,                 # warmup_ratio=0.1 ki jagah ye use karo
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_total_limit=1,
    report_to="none",
    seed=42,
)

trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# ==========================================
# 6. Train aur Final Test Set par Evaluate karo
# ==========================================
trainer.train()

test_output = trainer.predict(test_dataset)
y_pred = np.argmax(test_output.predictions, axis=1)

print("\n=== Final BERT Test Set Report ===")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Model aur tokenizer ko save kar lo (baad mein download/use karne ke liye)
model.save_pretrained("./commentlens_bert_final")
tokenizer.save_pretrained("./commentlens_bert_final")

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

# ==========================================
# 1. Setup & Device (CPU aur Kaggle GPU dono par chalega)
# ==========================================
MODEL_NAME = "google/muril-base-cased"
# Faster alternative: "distilbert-base-multilingual-cased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Running on device: {device}")

# ==========================================
# 2. Data Prep
# ==========================================
# df = pd.read_csv("merged_comments.csv")

# Local CPU par sirf code test karne ke liye neeche wali line uncomment kar lena:
# df = df.sample(1000, random_state=42).reset_index(drop=True)

x = df["comments"].astype(str)
labels = df["category"].astype(str).str.strip().str.lower()
label_mapping = {"neutral": 0, "positive": 1, "negative": 2}
y = labels.map(label_mapping)

x_train_full, x_test, y_train_full, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.1, random_state=42, stratify=y_train_full
)

# ==========================================
# 3. Tokenizer, Dataset aur DataLoaders
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
        )
        encoding["labels"] = int(self.labels[idx])
        return encoding

collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    CommentDataset(x_train, y_train, tokenizer),
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    CommentDataset(x_val, y_val, tokenizer),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    CommentDataset(x_test, y_test, tokenizer),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn,
)

# ==========================================
# 4. Model, Balanced Loss, Optimizer & Scheduler
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True,
).to(device)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=y_train.to_numpy(),
)
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

EPOCHS = 3
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler("cuda") if use_amp else None

# ==========================================
# 5. Evaluation Helper
# ==========================================
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}

            if use_amp:
                with torch.amp.autocast("cuda"):
                    logits = model(**batch).logits
            else:
                logits = model(**batch).logits

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return np.array(all_labels), np.array(all_preds), acc, macro_f1

# ==========================================
# 6. Pure PyTorch Training Loop (No accelerate)
# ==========================================
best_val_f1 = 0.0
save_dir = "./commentlens_bert_final"

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for step, batch in enumerate(train_loader, start=1):
        optimizer.zero_grad()
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}

        if use_amp:
            with torch.amp.autocast("cuda"):
                logits = model(**batch).logits
                loss = loss_fn(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(**batch).logits
            loss = loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()
        total_loss += loss.item()

        if step % 100 == 0 or step == len(train_loader):
            print(
                f"Epoch {epoch}/{EPOCHS} | Step {step}/{len(train_loader)} | "
                f"Avg Loss: {total_loss / step:.4f} | Time: {time.time() - t0:.1f}s"
            )

    _, _, val_acc, val_f1 = evaluate(model, val_loader)
    print(f"--> Epoch {epoch} Validation | Accuracy: {val_acc:.4f} | Macro F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"    Saved new best model (Macro F1: {best_val_f1:.4f})")

# ==========================================
# 7. Best Model Load karke Final Test Report
# ==========================================
best_model = AutoModelForSequenceClassification.from_pretrained(save_dir).to(device)
y_true, y_pred, test_acc, test_f1 = evaluate(best_model, test_loader)

print(f"\n=== Final BERT Test Set Report (Acc: {test_acc:.4f}, Macro F1: {test_f1:.4f}) ===")
print(classification_report(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

In [ ]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model="./commentlens_bert_final",
    tokenizer="./commentlens_bert_final",
    device=0 if torch.cuda.is_available() else -1,
)

id_to_label = {"LABEL_0": "Neutral", "LABEL_1": "Positive", "LABEL_2": "Negative"}

samples = [
    "bhai video ekdum mast thi par audio thoda low tha",
    "waste of time, kuch samajh nahi aaya",
    "next video kab aayegi?",
]

for text, res in zip(samples, clf(samples)):
    print(f"{text}  -->  {id_to_label[res['label']]} ({res['score']:.2f})")

In [5]:
from lightgbm import LGBMClassifier
model = LGBMClassifier(
    class_weight="balanced",
    n_estimators=400,
    learning_rate=0.08,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
)
model.fit(x_train , y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.446268 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 58770
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 5145
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


,learning_rate,0.08
,n_estimators,400
,class_weight,'balanced'
,colsample_bytree,0.8
,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,min_split_gain,0.0


In [6]:
import os
import zipfile
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# # 1. Unzip & Load Saved Model + Tokeniser
# # ==========================================
# MODEL_DIR = "./commentlens_muril_final"
# ZIP_PATH = "commentlens_muril_final.zip"

# model_path = os.path.abspath(MODEL_DIR)
# zip_path = os.path.abspath(ZIP_PATH)

# if not os.path.isfile(os.path.join(model_path, "config.json")) and os.path.isfile(zip_path):
#     os.makedirs(model_path, exist_ok=True)
#     with zipfile.ZipFile(zip_path, "r") as zf:
#         zf.extractall(model_path)

# if os.path.isfile(os.path.join(model_path, "config.json")):
#     model_config_path = os.path.join(model_path, "config.json")
# else:
#     model_config_path = None
#     for root, _, files in os.walk(model_path):
#         if "config.json" in files:
#             model_config_path = os.path.join(root, "config.json")
#             break

# if model_config_path is None:
#     raise FileNotFoundError(
#         f"Could not find config.json in {model_path} or its ZIP archive. "
#         f"Check that the model files exist and ZIP_PATH points to the correct archive."
#     )

model_path = '/mnt/WindowsD/Projects/commentlens/model' #os.path.dirname(model_config_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

ID_TO_LABEL = {0: "Neutral", 1: "Positive", 2: "Negative"}


# ==========================================
# 2. Batched Comment Prediction Helper
# ==========================================
@torch.inference_mode()
def predict_comments(comments, batch_size=64, max_len=128):
    if not comments:
        return []

    cleaned = [str(c).strip() for c in comments]
    results = []

    for start in range(0, len(cleaned), batch_size):
        batch_texts = cleaned[start : start + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        ).to(device)

        if device.type == "cuda":
            with torch.amp.autocast("cuda", dtype=torch.float16):
                logits = model(**inputs).logits
        else:
            logits = model(**inputs).logits

        probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()

        for text, prob_row in zip(batch_texts, probs):
            pred_id = int(np.argmax(prob_row))
            results.append(
                {
                    "comment": text,
                    "sentiment": ID_TO_LABEL[pred_id],
                    "confidence": round(float(prob_row[pred_id]), 4),
                    "neutral_prob": round(float(prob_row[0]), 4),
                    "positive_prob": round(float(prob_row[1]), 4),
                    "negative_prob": round(float(prob_row[2]), 4),
                }
            )

    return results


# ==========================================
# 3. Video-Level Summary (For CommentLens Extension)
# ==========================================
def analyze_video_comments(comments, batch_size=64):
    predictions = predict_comments(comments, batch_size=batch_size)
    total = len(predictions)
    if total == 0:
        return {"total_comments": 0}

    counts = {"Positive": 0, "Negative": 0, "Neutral": 0}
    for item in predictions:
        counts[item["sentiment"]] += 1

    percentages = {
        label: round((count / total) * 100, 2) for label, count in counts.items()
    }

    return {
        "total_comments": total,
        "counts": counts,
        "percentages": percentages,
        "predictions": predictions,
    }


# ==========================================
# 4. Test on Sample English + Hinglish YouTube Comments
# ==========================================
sample_comments = [
    "Bhai explanation ekdum clear thi, poora concept samajh aa gaya!",
    "Complete waste of time, clickbait thumbnail and zero useful info.",
    "Part 2 kab tak upload hoga?",
    "Editing is top notch but background music is way too loud.",
    "Ye trick sach mein kaam karti hai, thanks a lot bro!",
    "Kitna bakwas audio hai, kuch sunai hi nahi de raha.",
    "Timestamp 04:20 par jo formula bataya wo exam mein aaya tha.",
    "Not bad, decent overview for beginners.",
]

report = analyze_video_comments(sample_comments)

print("=== Video Sentiment Summary ===")
print(f"Total Comments: {report['total_comments']}")
print(f"Counts        : {report['counts']}")
print(f"Percentages   : {report['percentages']}\n")

df_results = pd.DataFrame(report["predictions"])
print(df_results[["comment", "sentiment", "confidence", "positive_prob", "negative_prob", "neutral_prob"]].to_string(index=False))

/mnt/WindowsD/Projects/commentlens/.commentlens/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1814.31it/s]


=== Video Sentiment Summary ===
Total Comments: 8
Counts        : {'Positive': 4, 'Negative': 1, 'Neutral': 3}
Percentages   : {'Positive': 50.0, 'Negative': 12.5, 'Neutral': 37.5}

                                                          comment sentiment  confidence  positive_prob  negative_prob  neutral_prob
  Bhai explanation ekdum clear thi, poora concept samajh aa gaya!  Positive      0.7844         0.7844         0.1277        0.0879
Complete waste of time, clickbait thumbnail and zero useful info.  Positive      0.8045         0.8045         0.1621        0.0334
                                      Part 2 kab tak upload hoga?   Neutral      0.9058         0.0385         0.0557        0.9058
       Editing is top notch but background music is way too loud.  Positive      0.7591         0.7591         0.2158        0.0251
             Ye trick sach mein kaam karti hai, thanks a lot bro!  Positive      0.8127         0.8127         0.1365        0.0508
              Kitna bakwas

In [7]:
import pickle

vectorizer = pickle.load(open('/mnt/WindowsD/Projects/commentlens/model/vectorizer.pkl' , 'rb'))

In [10]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Check MuRIL's actual performance on the 20% test split
test_preds = predict_comments(x_test.index.astype(str).tolist(), batch_size=128)
label_to_id = {"Neutral": 0, "Positive": 1, "Negative": 2}
y_pred_muril = [label_to_id[p["sentiment"]] for p in test_preds]

print("=== MuRIL Test Set Report ===")
print(classification_report(y_test, y_pred_muril, target_names=["Neutral (0)", "Positive (1)", "Negative (2)"]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_muril))

# 2. Compare LightGBM baseline on the exact same 8 sample comments
lgb_model = LGBMClassifier(
    class_weight="balanced",
    n_estimators=400,
    learning_rate=0.08,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
)
lgb_model.fit(x_train, y_train)
lgb_sample_preds = lgb_model.predict(vectorizer.transform(sample_comments))
id_to_label = {0: "Neutral", 1: "Positive", 2: "Negative"}

print("\n=== LightGBM vs MuRIL on Sample Comments ===")
for text, muril_res, lgb_id in zip(sample_comments, report["predictions"], lgb_sample_preds):
    print(f"Comment : {text}")
    print(f"  MuRIL   : {muril_res['sentiment']} ({muril_res['confidence']})")
    print(f"  LightGBM: {id_to_label[int(lgb_id)]}\n")

=== MuRIL Test Set Report ===
              precision    recall  f1-score   support

 Neutral (0)       0.33      1.00      0.50      3154
Positive (1)       0.00      0.00      0.00      3154
Negative (2)       0.00      0.00      0.00      3154

    accuracy                           0.33      9462
   macro avg       0.11      0.33      0.17      9462
weighted avg       0.11      0.33      0.17      9462

Confusion Matrix:
 [[3154    0    0]
 [3154    0    0]
 [3154    0    0]]


/mnt/WindowsD/Projects/commentlens/.commentlens/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/WindowsD/Projects/commentlens/.commentlens/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/WindowsD/Projects/commentlens/.commentlens/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.431570 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 58770
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 5145
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612

=== LightGBM vs MuRIL on Sample Comments ===
Comment : Bhai explanation ekdum clear thi, poora concept samajh aa gaya!
  MuRIL   : Positive (0.7844)
  LightGBM: Positive

Comment : Complete waste of time, clickbait thumbnail and zero useful info.
  MuRIL   : Positive (0.8045)
  LightGBM: Positive

Comment : Part 2 kab tak upload hoga?
  MuRIL   : Neutral (0.9058)
  LightGBM: Neutral

Comment : Editing is top notch but background music is way too loud.
  MuRIL   : Positive (0.7

/mnt/WindowsD/Projects/commentlens/.commentlens/lib/python3.14/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [3]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Load raw text comments (not the pre-vectorized 6000-col CSV)
df = pd.read_csv("/mnt/WindowsD/Projects/commentlens/data/interim/comments.csv")[["comments", "category"]].dropna()
x = df["comments"].astype(str)
y = df["category"].astype(str).str.strip().str.lower().map({"neutral": 0, "positive": 1, "negative": 2})

x_train_raw, x_test_raw, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = CountVectorizer(max_features=6000, ngram_range=(1, 2), dtype=np.float32)
X_train_vec = vectorizer.fit_transform(x_train_raw)
X_test_vec = vectorizer.transform(x_test_raw)

lgb_model = LGBMClassifier(
    class_weight="balanced",
    n_estimators=400,
    learning_rate=0.08,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
lgb_model.fit(X_train_vec, y_train)

print("=== Clean LightGBM Test Report ===")
print(classification_report(y_test, lgb_model.predict(X_test_vec), target_names=["Neutral (0)", "Positive (1)", "Negative (2)"]))

id_to_label = {0: "Neutral", 1: "Positive", 2: "Negative"}
sample_comments = [
    "Bhai explanation ekdum clear thi, poora concept samajh aa gaya!",
    "Complete waste of time, clickbait thumbnail and zero useful info.",
    "Part 2 kab tak upload hoga?",
    "Editing is top notch but background music is way too loud.",
    "Ye trick sach mein kaam karti hai, thanks a lot bro!",
    "Kitna bakwas audio hai, kuch sunai hi nahi de raha.",
    "Timestamp 04:20 par jo formula bataya wo exam mein aaya tha.",
    "Not bad, decent overview for beginners.",
]
lgb_preds = lgb_model.predict(vectorizer.transform(sample_comments))
for text, pred_id in zip(sample_comments, lgb_preds):
    print(f"{id_to_label[int(pred_id)]:<8} | {text}")

=== Clean LightGBM Test Report ===
              precision    recall  f1-score   support

 Neutral (0)       0.85      0.97      0.91      2529
Positive (1)       0.92      0.85      0.88      3154
Negative (2)       0.82      0.78      0.80      1650

    accuracy                           0.87      7333
   macro avg       0.86      0.86      0.86      7333
weighted avg       0.87      0.87      0.87      7333

Positive | Bhai explanation ekdum clear thi, poora concept samajh aa gaya!
Positive | Complete waste of time, clickbait thumbnail and zero useful info.
Neutral  | Part 2 kab tak upload hoga?
Positive | Editing is top notch but background music is way too loud.
Positive | Ye trick sach mein kaam karti hai, thanks a lot bro!
Neutral  | Kitna bakwas audio hai, kuch sunai hi nahi de raha.
Neutral  | Timestamp 04:20 par jo formula bataya wo exam mein aaya tha.
Positive | Not bad, decent overview for beginners.


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual"
SAVE_DIR = "./commentlens_xlmr_final"

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Reorder CardiffNLP's pretrained head [0:Neg, 1:Neu, 2:Pos] -> your [0:Neu, 1:Pos, 2:Neg]
with torch.no_grad():
    reorder = [1, 2, 0]
    model.classifier.out_proj.weight.copy_(model.classifier.out_proj.weight[reorder])
    model.classifier.out_proj.bias.copy_(model.classifier.out_proj.bias[reorder])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 22320.76it/s]


In [8]:
import numpy as np
import pandas as pd
import torch

# 1. Sync config metadata with your reordered head
model.config.id2label = {0: "Neutral", 1: "Positive", 2: "Negative"}
model.config.label2id = {"Neutral": 0, "Positive": 1, "Negative": 2}
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_comments = [
    "Bhai explanation ekdum clear thi, poora concept samajh aa gaya!",
    "Complete waste of time, clickbait thumbnail and zero useful info.",
    "Part 2 kab tak upload hoga?",
    "Editing is top notch but background music is way too loud.",
    "Ye trick sach mein kaam karti hai, thanks a lot bro!",
    "Kitna bakwas audio hai, kuch sunai hi nahi de raha.",
    "Timestamp 04:20 par jo formula bataya wo exam mein aaya tha.",
    "Not bad, decent overview for beginners.",
]

@torch.inference_mode()
def test_xlmr(comments):
    inputs = tokenizer(
        comments,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    ).to(device)

    logits = model(**inputs).logits
    probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()

    rows = []
    for text, p in zip(comments, probs):
        pred_id = int(np.argmax(p))
        rows.append({
            "comment": text,
            "sentiment": model.config.id2label[pred_id],
            "confidence": round(float(p[pred_id]), 4),
            "neutral_prob": round(float(p[0]), 4),
            "positive_prob": round(float(p[1]), 4),
            "negative_prob": round(float(p[2]), 4),
        })
    return pd.DataFrame(rows)

print(test_xlmr(sample_comments).to_string(index=False))

                                                          comment sentiment  confidence  neutral_prob  positive_prob  negative_prob
  Bhai explanation ekdum clear thi, poora concept samajh aa gaya!  Positive      0.3608        0.2981         0.3608         0.3411
Complete waste of time, clickbait thumbnail and zero useful info.  Negative      0.9670        0.0165         0.0165         0.9670
                                      Part 2 kab tak upload hoga?   Neutral      0.7936        0.7936         0.1542         0.0522
       Editing is top notch but background music is way too loud.  Positive      0.4121        0.2638         0.4121         0.3241
             Ye trick sach mein kaam karti hai, thanks a lot bro!  Positive      0.6988        0.1796         0.6988         0.1216
              Kitna bakwas audio hai, kuch sunai hi nahi de raha.  Negative      0.5254        0.2848         0.1898         0.5254
     Timestamp 04:20 par jo formula bataya wo exam mein aaya tha.   Neutral 

In [9]:
stress_test_data = [
    # --- Educational ---
    ("Educational", "Best DSA playlist on YouTube, cracked my placement because of this!", "Positive"),
    ("Educational", "Sir aapne itne aasaan tarike se samjhaya ki ek baar mein dimaag mein baith gaya", "Positive"),
    ("Educational", "Itna fast padha rahe ho kuch palle nahi pad raha, bilkul bekar explanation", "Negative"),
    ("Educational", "Can you share the GitHub link for the notes shown at 12:30?", "Neutral"),

    # --- Gaming ---
    ("Gaming", "Bro that 1v4 clutch at the end was pure madness, absolute W gameplay!", "Positive"),
    ("Gaming", "Kya OP headshots maare bhai, maza aa gaya dekh ke 🔥🔥", "Positive"),
    ("Gaming", "Stop camping the whole match, boring gameplay and full scripted lobby", "Negative"),
    ("Gaming", "Which graphics card and sensitivity settings are you using here?", "Neutral"),

    # --- Entertainment / Reviews ---
    ("Entertainment", "Finally an Indian web series with solid writing and zero overacting", "Positive"),
    ("Entertainment", "Paisa aur time dono barbaad, isse zyada cringe video aaj tak nahi dekha", "Negative"),
    ("Entertainment", "Thumbnail dekh ke aaya tha par video mein kuch aur hi chal raha hai", "Negative"),
    ("Entertainment", "Background song ka naam kya hai jo 03:15 par baja?", "Neutral"),

    # --- Political / News ---
    ("Political", "Rare to see unbiased journalism backed by actual data and RTI reports.", "Positive"),
    ("Political", "Poora video propaganda hai, ek hi side ki baat kar rahe ho bina facts ke", "Negative"),
    ("Political", "The new GST slab changes will take effect from next quarter as per the notification.", "Neutral"),
    ("Political", "Both parties have failed to address unemployment, just empty promises every election.", "Negative"),

    # --- Edge Cases: Pure Hinglish, Sarcasm, Emojis, Devanagari, Negation ---
    ("Pure Hinglish", "Bhai dil jeet liya, ekdum jhakaas video banayi hai!", "Positive"),
    ("Pure Hinglish", "Kya faltu bakchodi chal rahi hai, unsubscribe kar raha hu", "Negative"),
    ("Sarcasm", "Waah kya acting ki hai, Oscar de do inko koi 💀😂", "Negative"),
    ("Sarcasm", "Yeah sure, 20 minutes of talking just to say 'buy my course'. Brilliant.", "Negative"),
    ("Emoji Only", "🔥🔥🔥❤️❤️", "Positive"),
    ("Emoji Only", "🤮👎", "Negative"),
    ("Devanagari", "यह वीडियो बहुत ही जानकारीपूर्ण और शानदार है!", "Positive"),
    ("Negation Flip", "Not gonna lie, I expected it to be boring but this actually blew my mind.", "Positive"),
]

categories = [row[0] for row in stress_test_data]
comments = [row[1] for row in stress_test_data]
expected = [row[2] for row in stress_test_data]

res_df = test_xlmr(comments)
res_df.insert(0, "category", categories)
res_df["expected"] = expected
res_df["correct"] = res_df["sentiment"] == res_df["expected"]

overall_acc = res_df["correct"].mean() * 100
print(f"=== Stress Test Accuracy: {res_df['correct'].sum()}/{len(res_df)} ({overall_acc:.1f}%) ===\n")

print("--- Category-wise Breakdown ---")
print(res_df.groupby("category")["correct"].agg(["sum", "count", "mean"]).rename(columns={"sum": "correct", "count": "total", "mean": "accuracy"}))

print("\n--- Full Results ---")
print(res_df[["category", "expected", "sentiment", "confidence", "correct", "comment"]].to_string(index=False))

mistakes = res_df[~res_df["correct"]]
if not mistakes.empty:
    print("\n--- Mistakes to Inspect ---")
    print(mistakes[["category", "expected", "sentiment", "confidence", "neutral_prob", "positive_prob", "negative_prob", "comment"]].to_string(index=False))

=== Stress Test Accuracy: 18/24 (75.0%) ===

--- Category-wise Breakdown ---
               correct  total  accuracy
category                               
Devanagari           1      1      1.00
Educational          2      4      0.50
Emoji Only           2      2      1.00
Entertainment        3      4      0.75
Gaming               4      4      1.00
Negation Flip        0      1      0.00
Political            4      4      1.00
Pure Hinglish        1      2      0.50
Sarcasm              1      2      0.50

--- Full Results ---
     category expected sentiment  confidence  correct                                                                               comment
  Educational Positive  Positive      0.9883     True                   Best DSA playlist on YouTube, cracked my placement because of this!
  Educational Positive  Negative      0.3488    False       Sir aapne itne aasaan tarike se samjhaya ki ek baar mein dimaag mein baith gaya
  Educational Negative  Positive      0.3

In [10]:
import time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# 1. Model Load + Label Head Reorder
# ==========================================
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual"
DATA_PATH = "/mnt/WindowsD/Projects/commentlens/data/interim/comments.csv"
BATCH_SIZE = 128   # T4 GPU ke liye fast; agar CPU par chalao toh 32 kar dena
MAX_LEN = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")
if device.type == "cpu":
    print("Warning: CPU par poore CSV mein kaafi time lagega. Colab T4 GPU recommend hai.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# CardiffNLP [0:Neg, 1:Neu, 2:Pos] -> tumhara [0:Neu, 1:Pos, 2:Neg]
with torch.no_grad():
    reorder = [1, 2, 0]
    model.classifier.out_proj.weight.copy_(model.classifier.out_proj.weight[reorder])
    model.classifier.out_proj.bias.copy_(model.classifier.out_proj.bias[reorder])

model.config.id2label = {0: "Neutral", 1: "Positive", 2: "Negative"}
model.config.label2id = {"Neutral": 0, "Positive": 1, "Negative": 2}
model = model.to(device)
model.eval()

# ==========================================
# 2. Poora comments.csv Load Karo
# ==========================================
df = pd.read_csv(DATA_PATH)[["comments", "category"]].dropna().reset_index(drop=True)
texts = df["comments"].astype(str).tolist()

label_mapping = {
    "neutral": 0, "positive": 1, "negative": 2,
    "0": 0, "1": 1, "2": 2,
    "0.0": 0, "1.0": 1, "2.0": 2,
}
y_true = df["category"].astype(str).str.strip().str.lower().map(label_mapping)

if y_true.isna().any():
    invalid = df.loc[y_true.isna(), "category"].unique()
    raise ValueError(f"Unknown category labels found: {invalid}")

y_true = y_true.to_numpy(dtype=np.int64)
print(f"Total comments to evaluate: {len(texts)}")

# ==========================================
# 3. Fast Length-Sorted Batched Inference
# ==========================================
# Same length wale comments ek batch mein aayenge toh GPU par 2x fast chalega
order = np.argsort([len(t) for t in texts])
y_pred = np.empty(len(texts), dtype=np.int64)
confidences = np.empty(len(texts), dtype=np.float32)

t0 = time.time()
with torch.inference_mode():
    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Evaluating XLM-R"):
        batch_idx = order[start : start + BATCH_SIZE]
        batch_texts = [texts[i] for i in batch_idx]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        ).to(device)

        if device.type == "cuda":
            with torch.amp.autocast("cuda", dtype=torch.float16):
                logits = model(**inputs).logits
        else:
            logits = model(**inputs).logits

        probs = torch.softmax(logits.float(), dim=-1)
        conf, preds = torch.max(probs, dim=-1)

        y_pred[batch_idx] = preds.cpu().numpy()
        confidences[batch_idx] = conf.cpu().numpy()

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s ({len(texts) / elapsed:.1f} comments/sec)")

# ==========================================
# 4. Overall Accuracy, Classification Report & Disagreements
# ==========================================
acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"\n=== XLM-RoBERTa Full comments.csv Report ===")
print(f"Overall Accuracy : {acc * 100:.2f}%")
print(f"Macro F1 Score   : {macro_f1:.4f}\n")
print(classification_report(y_true, y_pred, target_names=["Neutral (0)", "Positive (1)", "Negative (2)"]))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# Dekhne ke liye ki jahan XLM-R aur CSV ka label alag hai wahan kaun sahi hai
id2label = {0: "Neutral", 1: "Positive", 2: "Negative"}
mismatch_df = pd.DataFrame({
    "comment": texts,
    "csv_label": [id2label[i] for i in y_true],
    "xlmr_pred": [id2label[i] for i in y_pred],
    "confidence": np.round(confidences, 3),
})
mismatches = mismatch_df[mismatch_df["csv_label"] != mismatch_df["xlmr_pred"]]

print(f"\nTotal mismatches: {len(mismatches)} / {len(texts)}")
print("\n=== Top 15 High-Confidence Disagreements (Khud padh kar dekho kaun sahi hai) ===")
print(mismatches.sort_values("confidence", ascending=False).head(15).to_string(index=False))

Running on: cpu


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 31216.17it/s]


Total comments to evaluate: 36662


Evaluating XLM-R: 100%|██████████| 287/287 [20:36<00:00,  4.31s/it]


Done in 1236.3s (29.7 comments/sec)

=== XLM-RoBERTa Full comments.csv Report ===
Overall Accuracy : 47.20%
Macro F1 Score   : 0.4672

              precision    recall  f1-score   support

 Neutral (0)       0.46      0.58      0.51     12644
Positive (1)       0.68      0.26      0.38     15770
Negative (2)       0.40      0.71      0.51      8248

    accuracy                           0.47     36662
   macro avg       0.51      0.52      0.47     36662
weighted avg       0.54      0.47      0.45     36662

Confusion Matrix:
 [[7316 1614 3714]
 [6635 4113 5022]
 [2070  304 5874]]

Total mismatches: 19359 / 36662

=== Top 15 High-Confidence Disagreements (Khud padh kar dekho kaun sahi hai) ===
                                                                                                                                                                                                                                                                                             comment c